In [ ]:
# %%
# SECTION 1: Imports
# -------------------
# Loads all required libraries:
#   - pandas: reads Excel files and performs DataFrame merges
#   - pathlib.Path: resolves file paths for Excel export
#
# Requirements: pip install pandas openpyxl

from pathlib import Path

import pandas as pd

print("Imports loaded successfully.")

In [ ]:
# %%
# SECTION 2: Configuration
# -------------------------
# Set your input files and merge options here before running any section below.
#
# input_files  - List of Excel file paths to merge (minimum 2, order matters for chained merges)
# merge_key    - Column name to merge on (e.g. 'BACID'). Set to None to merge on all shared columns.
# sheet        - Sheet name or index to read from each file (0 = first sheet)
# output_path  - Where to save the final merged Excel file

input_files = [
    'AdamsPullACT_ARC.xlsx',
    'IansPullAct_Arch_4.xlsx',
    # Add more files here for chained merges:
    # 'ThirdFile.xlsx',
]
merge_key   = 'BACID'   # e.g. 'BACID', or None to merge on all common columns
sheet       = 0         # 0 = first sheet, or use sheet name e.g. 'Sheet1'
output_path = 'merged_output.xlsx'

print(f"Config ready. Files to merge: {input_files}")
print(f"Merge key: {merge_key if merge_key else 'all common columns'}")

In [ ]:
# %%
# SECTION 3: load_files()
# ------------------------
# Reads each Excel file listed in input_files into a pandas DataFrame.
# Prints the filename, row count, and column count for each file so you can
# confirm the data loaded correctly before proceeding.
# Results are stored in 'named_dfs' — a list of (filename, DataFrame) tuples.

def load_files(file_paths, sheet):
    results = []
    for path in file_paths:
        df = pd.read_excel(path, sheet_name=sheet)
        print(f"Loaded '{Path(path).name}': {df.shape[0]} rows x {df.shape[1]} cols")
        results.append((Path(path).name, df))
    return results

named_dfs = load_files(input_files, sheet)

In [ ]:
# %%
# SECTION 4: reset_indexes()
# ---------------------------
# Resets the index on each DataFrame to clean integer indexes (0, 1, 2, ...).
# This ensures the merge works correctly regardless of how the source Excel
# files were originally indexed. Prints the first 3 rows of each file as a
# preview so you can verify column names and data types before merging.

def reset_indexes(named_dfs):
    return [(name, df.reset_index(drop=True)) for name, df in named_dfs]

named_dfs = reset_indexes(named_dfs)

print("Indexes reset. Preview of each DataFrame:")
for name, df in named_dfs:
    print(f"\n--- {name} ---")
    print(df.head(3))

In [ ]:
# %%
# SECTION 5: merge_dataframes()
# ------------------------------
# Chains outer merges across all DataFrames in the order they appear in input_files.
# Starts with the first file and merges each subsequent file onto the result.
# Each merge step adds a '_merge_N' indicator column with values:
#   'left_only'  — row exists in the left/accumulated result but not in the new file
#   'right_only' — row exists in the new file but not in the accumulated result
#   'both'       — row exists in both sources
# Prints the row count after each step so you can track how the dataset grows.

def merge_dataframes(named_dfs, merge_key):
    _, result = named_dfs[0]
    result = result.copy()
    for i, (name, df) in enumerate(named_dfs[1:], start=1):
        indicator_col = f'_merge_{i}'
        kwargs = {'how': 'outer', 'indicator': indicator_col}
        if merge_key:
            kwargs['on'] = merge_key
        result = pd.merge(result, df, **kwargs)
        print(f"After merge {i} (+ '{name}'): {result.shape[0]} rows")
    return result

merged = merge_dataframes(named_dfs, merge_key)
print(f"\nFinal merged shape: {merged.shape}")

In [ ]:
# %%
# SECTION 6: compare_dataframes()
# --------------------------------
# Reads the '_merge_N' indicator columns added during Section 5 and prints a
# summary of what's shared vs. unique at each merge step:
#   'both'       — rows that matched between the two sources at that step
#   'left_only'  — rows only in the accumulated left side
#   'right_only' — rows only in the new file added at that step
# Use this to quickly understand overlap and gaps between your Excel sources
# before committing to the export.

def compare_dataframes(merged):
    merge_cols = [c for c in merged.columns if c.startswith('_merge_')]
    summary = {}
    for col in merge_cols:
        counts = merged[col].value_counts().to_dict()
        summary[col] = {
            'left_only':  counts.get('left_only', 0),
            'right_only': counts.get('right_only', 0),
            'both':       counts.get('both', 0),
        }
    return summary

summary = compare_dataframes(merged)
print("--- Comparison Summary ---")
for merge_step, counts in summary.items():
    print(f"  {merge_step}: {counts['both']} shared | "
          f"{counts['left_only']} left-only | {counts['right_only']} right-only")

In [ ]:
# %%
# SECTION 7: Inspect result
# --------------------------
# Prints the total row count and a preview of the final merged DataFrame.
# Review this before exporting — check that the _merge indicator columns look
# correct and that the data makes sense. If something looks wrong, adjust the
# merge_key or file order in Section 2 and re-run from Section 5.

print(f"Total rows in merged result: {merged.shape[0]}")
print(f"Columns: {list(merged.columns)}")
print("\nPreview (first 5 rows):")
merged.head(5)

In [ ]:
# %%
# SECTION 8: export_to_excel()
# -----------------------------
# Writes the final merged DataFrame to an Excel file at output_path (set in Section 2).
# The output includes all original columns plus the '_merge_N' indicator columns
# so you can filter and review the results in Excel.
# Run this last, after confirming the Section 7 preview looks correct.

def export_to_excel(df, output_path):
    df.to_excel(output_path, index=False)
    print(f"Exported {df.shape[0]} rows to '{output_path}'")

export_to_excel(merged, output_path)